# 03 — SHAP Analysis: Model Interpretability

SHAP (SHapley Additive exPlanations) values decompose each prediction into per-feature contributions.  
This is critical for credit risk — regulators (SR 11-7, ECOA) require explainability of credit decisions.

In [ ]:
import pandas as pd
import numpy as np
import json
import pickle
import shap
import matplotlib.pyplot as plt
from pathlib import Path

GOLD_DIR = Path("../data/gold")
MODELS_DIR = Path("../data/models")

# Load model
with open(MODELS_DIR / "champion" / "model.pkl", "rb") as f:
    model = pickle.load(f)
with open(GOLD_DIR / "feature_metadata.json") as f:
    meta = json.load(f)

feature_cols = meta["feature_columns"]
test = pd.read_parquet(GOLD_DIR / "features_test.parquet")
X_test = test[feature_cols]
y_test = test["default"]

# Use a sample for SHAP (full dataset is too slow)
sample_idx = np.random.RandomState(42).choice(len(X_test), size=5000, replace=False)
X_sample = X_test.iloc[sample_idx]

print(f"Computing SHAP values for {len(X_sample):,} samples...")

In [ ]:
# Compute SHAP values
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_sample)

## 1. SHAP Summary Plot — Global Feature Importance

In [ ]:
# Beeswarm plot — shows direction and magnitude of each feature's impact
shap.summary_plot(shap_values, X_sample, max_display=20, show=False)
plt.title("SHAP Summary — Top 20 Features", fontweight='bold')
plt.tight_layout()
plt.show()

## 2. SHAP Bar Plot — Mean Absolute Impact

In [ ]:
# Bar plot — mean |SHAP value| per feature
shap.summary_plot(shap_values, X_sample, plot_type="bar", max_display=20, show=False)
plt.title("Mean |SHAP Value| — Feature Importance Ranking", fontweight='bold')
plt.tight_layout()
plt.show()

## 3. SHAP Dependence Plots — Key Features

In [ ]:
# Dependence plots for top 4 features
top_4 = pd.DataFrame({'feature': feature_cols, 'importance': np.abs(shap_values).mean(axis=0)}) \
    .sort_values('importance', ascending=False).head(4)['feature'].tolist()

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for i, feat in enumerate(top_4):
    shap.dependence_plot(feat, shap_values, X_sample, ax=axes[i], show=False)
    axes[i].set_title(feat, fontweight='bold')

plt.suptitle("SHAP Dependence Plots — Top 4 Features", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Individual Explanation — Waterfall Plot

Show how a single credit decision was made (adverse action explanation).

In [ ]:
# Pick a declined applicant (high score) for waterfall explanation
scores = model.predict_proba(X_sample)[:, 1]
declined_idx = np.argmax(scores)  # highest risk applicant in sample

print(f"Applicant score: {scores[declined_idx]:.4f} (DECLINE)")
print(f"Actual outcome: {'DEFAULT' if y_test.iloc[sample_idx[declined_idx]] == 1 else 'PAID'}\n")

# Waterfall plot
shap_explanation = shap.Explanation(
    values=shap_values[declined_idx],
    base_values=explainer.expected_value,
    data=X_sample.iloc[declined_idx],
    feature_names=feature_cols,
)
shap.waterfall_plot(shap_explanation, max_display=15, show=False)
plt.title("Adverse Action Explanation — Declined Applicant", fontweight='bold')
plt.tight_layout()
plt.show()